# Долгосрочная память аптечного AI-ассистента · этап 4: справочник ГРЛС и проверка безопасности в коде

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/serge3emskov/pharma-ltm-assistant/blob/main/notebooks/04_reference_safety.ipynb)

На этапе 3 самым слабым местом оказались ответы: модель знает из памяти, что покупатель водит машину, и всё равно советует Супрастин. Этап 4 переносит то, что можно проверить детерминированно, в код — так же, как на этапах 1–2 в код ушло управление памятью.

| Что | Как |
|---|---|
| Справочник | торговое наименование → МНН, состав, путь введения, отпуск по рецепту; стартовый набор (70 МНН, с кодами АТХ) + выгрузка ГРЛС |
| Классы | по префиксам АТХ и спискам МНН: НПВП, антикоагулянты, антигистаминные 1 поколения… (`safety_data/classes.json`) |
| Правила | взаимодействия, заболевания, беременность и ГВ, возраст, образ жизни, аллергии, дублирование действующего вещества, рецептурный отпуск (`safety_data/rules.json`) |
| До ответа | препараты из вопроса проверяются по профилю из памяти; результат и ограничения по профилю — в контексте модели |
| После ответа | препараты из ответа модели проверяются ещё раз; если ответ не называет причину, предупреждение дописывает код |

**Что нужно.** Разделы 1–6 работают без GPU. Для раздела 7 (ответы модели) — Среда выполнения → Сменить среду выполнения → **T4 GPU**; ответы кэшируются на Google Drive вместе с кэшем этапа 3, поэтому ответы «как на этапе 3» берутся из кэша.

**Выгрузка ГРЛС** — набор файлов xlsx по статусу записи: «Действующий», «Изменённый», «Истёкший», «Исключённый» и другие (`grls<дата>-…xlsx`). Положите их все в папку `pharma-ltm/grls/` на Google Drive: ноутбук прочитает их вместе и пропустит исключённые записи. Кодов АТХ в выгрузке нет — классы препаратов из ГРЛС определяются по МНН, местные формы — по колонке «Формы выпуска», оттуда же берётся отпуск по рецепту. Без файлов ноутбук работает на стартовом справочнике.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/serge3emskov/pharma-ltm-assistant.git"

if IN_COLAB:
    if os.path.exists("pharma-ltm-assistant"):
        subprocess.run(["git", "-C", "pharma-ltm-assistant", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    ROOT = os.path.abspath("pharma-ltm-assistant")
    for req in ("requirements.txt", "requirements-data.txt", "requirements-llm.txt"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/{req}"], check=True)
else:
    ROOT = os.path.abspath("..")

SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
# После git pull Python не перечитывает уже импортированный пакет — выгружаем старую версию
for name in [m for m in sys.modules if m == "pharmaltm" or m.startswith("pharmaltm.")]:
    del sys.modules[name]
import pharmaltm

commit = subprocess.run(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"pharmaltm {pharmaltm.__version__} · commit {commit or 'нет git'}")

## 1. Настройки

* `GRLS_FILES` — список файлов выгрузки; `None` — все `grls*` из `pharma-ltm/grls/` (если таких нет — все .xlsx/.xls/.csv/.zip в папке).
* `GRLS_COLUMNS` — если колонки не распознались по заголовкам: `{"trade_name": "…", "inn": "…", "form": "…"}` (текст заголовка или номер колонки с нуля).
* `MODEL_ID` — модель для раздела 7 (на этапе 3 выбрана Qwen3-4B).

In [ ]:
import gc, json, time
from collections import Counter
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from pharmaltm import CachedLLM, DrugCatalog, Normalizer, QwenTransformersLLM, SafetyChecker, load_grls
from pharmaltm.devcheck import load_dev
from pharmaltm.evaluation import (compare_reports, gold_backend, load_golden, run_golden, summarize, to_report,
                                  warm_cache)
from pharmaltm.catalog import CATALOG_VERSION
from pharmaltm.safety import LEVEL_LABELS
from pharmaltm.safety_eval import (load_cases, normalization_diff, replay_answers, run_cases, run_safety_e2e,
                                   summarize_cases, summarize_e2e)

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 240)

BACKEND = os.environ.get("PHARMALTM_BACKEND", "qwen")   # "gold" — проверка ноутбука без GPU
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
GRLS_FILES = None
GRLS_COLUMNS = None
BATCH_SIZE = 8
USE_DRIVE = IN_COLAB

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/pharma-ltm")
else:
    WORK = Path(ROOT) / "runs"
GRLS_DIR, CACHE_DIR, REPORT_DIR = WORK / "grls", WORK / "cache", WORK / "reports"
for d in (GRLS_DIR, CACHE_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("выгрузка ГРЛС:", GRLS_DIR, "| кэш ответов:", CACHE_DIR)

GOLDEN = load_golden()
DEV_DIALOGS, _ = load_dev()
CASES = load_cases()
REPORT = {"commit": commit, "created": datetime.now().isoformat(timespec="seconds"), "model": MODEL_ID}

## 2. Справочник: стартовый набор + ГРЛС

Загрузчик сам находит строку заголовка (над ней в выгрузке строка с датой реестра) и колонки по ключевым словам, пропускает строки-разделы, аннулированные записи и файл исключённых. Разобранный справочник сохраняется на Drive (`catalog_*.json.gz`): повторный запуск загружает его за секунды.

Чтобы загрузить файлы прямо из ноутбука, поставьте `UPLOAD = True`.

In [ ]:
import hashlib

UPLOAD = False
if UPLOAD and IN_COLAB:
    from google.colab import files
    for name, data in files.upload().items():
        (GRLS_DIR / name).write_bytes(data)

def find_grls() -> list:
    if GRLS_FILES:
        return [Path(p) for p in GRLS_FILES]
    supported = [p for p in sorted(GRLS_DIR.iterdir()) if p.suffix.lower() in (".zip", ".xlsx", ".xls", ".csv")]
    # Свежая выгрузка ГРЛС (grls<дата>-…) важнее старых наборов открытых данных, если лежат рядом.
    # Архив папки, скачанный с Drive, содержит те же файлы: при наличии таблиц архивы не берём
    fresh = [p for p in supported if p.name.lower().startswith("grls")]
    tables = [p for p in fresh if p.suffix.lower() != ".zip"]
    return tables or fresh or supported

src = find_grls()
LOAD_REPORT = None
if not src:
    print("Выгрузка ГРЛС не найдена — работаем на стартовом справочнике.")
    CATALOG = DrugCatalog()
else:
    print("файлы выгрузки:", *[f"  {p.name} ({p.stat().st_size / 1e6:.1f} МБ)" for p in src], sep="\n")
    # Ключ кэша — файлы выгрузки и версия правил сборки: после обновления кода справочник собирается заново
    key = hashlib.md5("|".join([f"v{CATALOG_VERSION}"] + [f"{p.name}:{p.stat().st_size}" for p in src])
                      .encode()).hexdigest()[:10]
    cache = GRLS_DIR / f"catalog_{key}.json.gz"
    CATALOG = None
    if cache.exists() and GRLS_COLUMNS is None:
        try:
            CATALOG = DrugCatalog.load(cache)
            print(f"справочник из кэша: {cache.name}")
        except ValueError as e:
            print(e)
    if CATALOG is None:
        t0 = time.time()
        products, LOAD_REPORT = load_grls(src, columns=GRLS_COLUMNS)
        CATALOG = DrugCatalog()
        CATALOG.add_products(products)
        CATALOG.save(cache)
        print(f"прочитано {len(products)} строк за {time.time() - t0:.0f} с → {cache.name}")
        display(pd.DataFrame([{"таблица": t["name"], "прочитана": t["used"], "причина": t["reason"],
                               "строка заголовка": t["header_row"], "колонки": t["columns"], "строк": t["rows"],
                               "препаратов": t["products"], "пропущено": t["skipped"]}
                              for t in LOAD_REPORT.as_dict()["tables"]]))
        unused = [t for t in LOAD_REPORT.tables if not t.used and t.reason != "таблица исключённых записей"]
        for t in unused:
            print(f"не распознана таблица {t.name}: {t.reason}; первая строка: {t.header[:8]}")
            print("  → задайте GRLS_COLUMNS по заголовкам выше и перезапустите ячейку")

display(pd.DataFrame([CATALOG.summary()]).T.rename(columns={0: "значение"}))
AMBIGUOUS = CATALOG.ambiguous(300)
RESOLVED = CATALOG.resolved(300)
SEED_CONFLICTS = CATALOG.seed_conflicts(300)
if RESOLVED:
    display(Markdown(f"**Наименования с разным составом в разных записях: выбран самый частый состав** "
                     f"({CATALOG.summary()['ambiguous_resolved']}, первые 20). Для проверки безопасности у наименования "
                     f"остаются вещества из всех составов"))
    display(pd.DataFrame([{"наименование": k, "выбран": w,
                           "строк ГРЛС по составам": " | ".join(f"{c} — {n}" for c, n in cs[:4])}
                          for k, w, cs in RESOLVED[:20]]))
if AMBIGUOUS:
    display(Markdown(f"**Не разрешены: составы встречаются одинаково часто** — не угадываем, считаем нераспознанными "
                     f"({len(CATALOG.ambiguous(100000))}, первые 20)"))
    display(pd.DataFrame([{"наименование": k, "составы по ГРЛС": " | ".join(v[:4])} for k, v in AMBIGUOUS[:20]]))
if SEED_CONFLICTS:
    display(Markdown(f"**Синонимы стартового набора, у которых в ГРЛС другой состав** — оставлен стартовый ключ "
                     f"({len(SEED_CONFLICTS)}, первые 20)"))
    display(pd.DataFrame([{"синоним": k, "ключ памяти": own, "по ГРЛС": " | ".join(v[:3])}
                          for k, own, v in SEED_CONFLICTS[:20]]))

NORM = Normalizer(catalog=CATALOG)
CHECKER = SafetyChecker(CATALOG)
REPORT["catalog"] = {"source": [p.name for p in src], "summary": CATALOG.summary(),
                     "load": LOAD_REPORT.as_dict() if LOAD_REPORT else None,
                     "ambiguous": AMBIGUOUS, "resolved": RESOLVED, "seed_conflicts": SEED_CONFLICTS}

In [ ]:
OTC_LABEL = {True: "без рецепта", False: "по рецепту", None: "—"}
rows = []
for name in ("Энап 5 мг", "Нурофен Экспресс", "Терафлю", "Детралекс", "Монурал", "Кагоцел", "Вольтарен Эмульгель",
             "Фастум гель", "Цефтриаксон", "Омез", "Валсакор", "Колдрекс", "Ингалипт", "Валидол", "Панангин"):
    info = CATALOG.resolve(name)
    rows.append({"название": name, "ключ (МНН)": info.canonical if info.matched else "не найдено",
                 "веществ для проверки": len(info.components),
                 "АТХ": ", ".join(info.atc) or "—", "пути введения": ", ".join(info.routes) or "—",
                 "отпуск": OTC_LABEL[info.otc],
                 "классы": ", ".join(sorted(CHECKER.classes_for(info.components, info.atc, info.routes))) or "—"})
pd.DataFrame(rows)

## 3. Что меняет большой справочник

* Названия из эталонной разметки (golden set, dev-набор): какие распознаёт только ГРЛС и не изменился ли ключ памяти.
* Упоминания препаратов в репликах покупателей, которые находит только ГРЛС. Сюда попадают и новые препараты, и обычные слова, совпавшие с торговым наименованием, — список стоит просмотреть глазами.
* Регрессия: эталонная разметка golden set через память со справочником ГРЛС. Ключи могут измениться только у названий, которых не было в стартовом справочнике («Детралекс» → гесперидин + диосмин): golden set размечен стартовым справочником, и для таких фактов эталонный ключ — само название. Изменение ключа у уже распознанного названия — регрессия.

In [ ]:
diff = normalization_diff(Normalizer(), NORM, GOLDEN, DEV_DIALOGS, CASES)
print(f"названий в разметке: {diff['names']} · распознано: стартовый {diff['matched_seed']}, с ГРЛС {diff['matched_full']}")
print("не распознано и с ГРЛС:", diff["unmatched_full"] or "—")
if diff["changed"]:
    display(Markdown("**Ключ памяти изменился** (стартовый → с ГРЛС)"))
    display(pd.DataFrame(diff["changed"]))
if diff["extra_mentions"]:
    display(Markdown("**Упоминания, которые находит только ГРЛС**"))
    display(pd.DataFrame(diff["extra_mentions"]).head(30))
REPORT["normalization"] = diff

In [ ]:
gold = summarize(run_golden(gold_backend(GOLDEN), GOLDEN, normalizer=NORM, run_answers=False))
regress = {"память: факты F1": gold["memory"]["facts"]["f1"], "статусы разделов F1": gold["memory"]["section_states"]["f1"],
           "retrieval": gold["retrieval"]["gold_query_types"]["probes_ok"] / gold["retrieval"]["gold_query_types"]["probes"]}
print(regress)
regressions = [r for r in diff["changed"] if r["seed"] != "—"]
REPORT["regression_gold"] = {**regress, "regressions": regressions}
if all(v == 1.0 for v in regress.values()):
    print("справочник ГРЛС не меняет память на эталонной разметке")
elif not regressions:
    print("ключи изменились только у названий, которых не было в стартовом справочнике: теперь ключ — МНН. Регрессий нет")
else:
    print("РЕГРЕССИЯ: изменился ключ у названий из стартового справочника:")
    display(pd.DataFrame(regressions))

## 4. Правила

Классы препаратов определяются по АТХ (стартовый набор) и по МНН (все препараты, включая ГРЛС). Местная форма исключается по АТХ или по путям введения из «Форм выпуска»: правило для таблеток НПВП не срабатывает на гель.

In [ ]:
rules = CHECKER.rules
display(pd.DataFrame(Counter((r.kind, LEVEL_LABELS[r.level]) for r in rules.rules).most_common(),
                     columns=["тип и уровень", "правил"]))
per_class = Counter()
for e in CATALOG.entries():
    for c in CHECKER.classes_for(e.components or (e.canonical,), e.atc, CATALOG.routes_for(e.canonical)):
        per_class[c] += 1
display(pd.DataFrame([{"класс": c.id, "название": c.label, "МНН и составов в справочнике": per_class.get(c.id, 0)}
                      for c in rules.classes]))

## 5. Проверочные случаи

41 случай: память покупателя + вопрос → полный список ожидаемых предупреждений. Лишнее предупреждение считается такой же ошибкой, как пропущенное; 6 случаев — контрольные, где предупреждений быть не должно (гель вместо таблеток, «Нурофен или Найз» как выбор, чужой профиль).

Случаи писались вместе с правилами, поэтому это регрессионный набор, а не независимая оценка.

In [ ]:
case_rows = run_cases(CHECKER, CASES, NORM)
REPORT["cases"] = {"summary": summarize_cases(case_rows), "rows": case_rows}
print(REPORT["cases"]["summary"])
pd.DataFrame([{"случай": r["id"], "что": r["title"], "ок": "✓" if r["ok"] else "✗",
               "предупреждения": "\n".join(r["alerts"]) or "—", "только по рецепту": ", ".join(r["rx_only"]),
               "не хватает": r["missing"] or "", "лишние": r["extra"] or ""} for r in case_rows])

## 6. Повтор ответов этапа 3

Ответы Qwen3-4B и Qwen3-8B из отчёта этапа 3 (прогон 2) проходят проверку после ответа. Память восстановлена эталонной разметкой. Модели генерировали эти ответы без блока проверки — видно, что код дописал бы к ним.

In [ ]:
stage3 = json.loads((Path(ROOT) / "reports" / "stage3_run2.json").read_text(encoding="utf-8"))
replay = replay_answers(stage3, GOLDEN, CHECKER, NORM)
REPORT["replay"] = replay
pd.DataFrame([{"модель": r["model"].split("/")[-1], "сценарий": r["scenario"], "вопрос": r["question"],
               "препараты в ответе": ", ".join(r["answer_drugs"]) or "—",
               "учтено моделью": len(r["covered"]) + len(r["declined"]),
               "дописал код": "\n".join(r["appended"]) or "—", "дозы в ответе": ", ".join(r["doses"]) or ""}
              for r in replay])

In [ ]:
example = next((r for r in replay if "Супрастин" in r["answer"]), None) or next((r for r in replay if r["appended"]), None)
if example:
    display(Markdown(f"**{example['model']} · {example['question']}**\n\nИтоговый ответ покупателю:"))
    print(example["final"])

## 7. Ответы модели: без проверки и с ней

Для 23 случаев с проверкой ответа модель отвечает дважды:

* **baseline** — контекст как на этапе 3, без блока проверки;
* **safety** — тот же контекст плюс блок «ПРОВЕРКА ПО СПРАВОЧНИКУ».

В обоих условиях к ответу применяется проверка после ответа. Метрики: эвристики ответа (написаны вручную, независимо от правил) до и после проверки; доля предупреждений, которые модель учла сама; сколько дописал код; ответы с дозами (промпт их запрещает).

Затем — golden set этапа 3 с проверкой. Память строится стартовым справочником, как на этапе 3 (write-path берётся из кэша), а проверка безопасности использует полный справочник: меняются только ответы.

In [ ]:
def gpu_ready() -> bool:
    if BACKEND != "qwen":
        return True
    try:
        import torch
        return torch.cuda.is_available()
    except ImportError:
        return False

LLM = None
if gpu_ready():
    if BACKEND == "qwen":
        t0 = time.time()
        base = QwenTransformersLLM(MODEL_ID, load_in_4bit=True)
        print(f"{MODEL_ID} загружена за {time.time() - t0:.0f} с")
    else:
        base = gold_backend(GOLDEN)
    LLM = CachedLLM(base, str(CACHE_DIR / f"{MODEL_ID.replace('/', '__')}.jsonl"))
else:
    print("GPU нет — раздел 7 пропущен. Среда выполнения → Сменить среду выполнения → T4 GPU.")

In [ ]:
if LLM is not None:
    t0 = time.time()
    e2e = run_safety_e2e(LLM, CHECKER, CASES, normalizer=NORM, batch_size=BATCH_SIZE)
    REPORT["e2e"] = {"summary": summarize_e2e(e2e), "rows": e2e}
    print(f"готово за {time.time() - t0:.0f} с")
    display(pd.DataFrame(REPORT["e2e"]["summary"]))

In [ ]:
if LLM is not None:
    by_case = {}
    for r in e2e:
        row = by_case.setdefault(r["id"], {"случай": r["id"], "что": r["title"]})
        mark = lambda checks: " ".join("✓" if c["passed"] else "✗" for c in checks)
        row[f"{r['condition']}: ответ"] = mark(r["checks_raw"])
        row[f"{r['condition']}: учтено/дописано"] = f"{len(r['covered']) + len(r['declined'])}/{len(r['appended'])}"
    display(pd.DataFrame(by_case.values()))

In [ ]:
if LLM is not None:
    SHOW = "s01"   # водитель и Супрастин
    for r in e2e:
        if r["id"] == SHOW:
            display(Markdown(f"**{r['condition']}** · {r['question']}"))
            print(r["final"], "\n")

In [ ]:
if LLM is not None:
    warm_cache(LLM, GOLDEN, batch_size=BATCH_SIZE)   # write-path этапа 3 — из кэша
    golden_runs = {}
    for label, safety in (("этап 3 (без проверки)", None), ("этап 4 (с проверкой)", CHECKER)):
        rep = to_report(run_golden(LLM, GOLDEN, safety=safety, batch_size=BATCH_SIZE), GOLDEN)
        golden_runs[label] = {"model": label, "summary": rep["summary"],
                              "probes": [p for p in rep["probes"] if p["answer"]]}
    REPORT["golden"] = golden_runs
    display(pd.DataFrame(compare_reports(list(golden_runs.values()))).set_index("метрика"))
    # Проверки ответа по пробам: где условия разошлись
    marks = lambda p: " ".join("✓" if c["passed"] else "✗" for c in p["answer_checks"])
    rows = {}
    for label, run in golden_runs.items():
        for p in run["probes"]:
            row = rows.setdefault((p["scenario"], p["text"]), {"сценарий": p["scenario"], "вопрос": p["text"]})
            row[label] = marks(p)
    display(pd.DataFrame(rows.values()))
    for p in golden_runs["этап 4 (с проверкой)"]["probes"]:
        if not all(c["passed"] for c in p["answer_checks"]):
            display(Markdown(f"**{p['scenario']} · {p['text']}** — не пройдено: "
                             + "; ".join(c["why"] for c in p["answer_checks"] if not c["passed"])))
            print(p["answer"])

## 8. Отчёт

Все результаты одним файлом — его можно прислать для разбора.

In [ ]:
out = WORK / f"stage4_{commit or 'local'}_{datetime.now():%Y%m%d_%H%M}.json"
out.write_text(json.dumps(REPORT, ensure_ascii=False, indent=1, default=list), encoding="utf-8")
print("сохранено:", out)
if IN_COLAB:
    from google.colab import files
    files.download(str(out))

> Демонстрационный проект. Правила упрощены и не заменяют инструкцию к препарату; данные синтетические, ответы ассистента не являются медицинской рекомендацией.